In [58]:
import pandas as pd

In [59]:
#Read in the base_1 tables
df = pd.read_csv("/home/katherine-schwind/Downloads/PUDF1Q2019_tab-delimited/PUDF_base1_1q2019_tab.txt",sep = "\t", low_memory=False)

In [60]:
# Drop fields 23-31,83-166, 168 (leaving penultimate column 167)
df = df.drop(columns=df.columns[82:166])
df = df.drop(columns=df.columns[22:31])
df = df.drop(columns=df.columns[-1])

In [61]:
# Combine SPEC_UNIT codes into one concatenated column
df['SPEC_UNIT'] = ''
for column in ['SPEC_UNIT_1', 'SPEC_UNIT_2', 'SPEC_UNIT_3', 'SPEC_UNIT_4', 'SPEC_UNIT_5']:
    df['SPEC_UNIT'] += df[column].fillna('')

In [62]:
# Drop the individual SPEC_UNIT columns
df = df.drop(columns = ['SPEC_UNIT_1', 'SPEC_UNIT_2', 'SPEC_UNIT_3', 'SPEC_UNIT_4', 'SPEC_UNIT_5'])

In [63]:
# Intialize DIAG_CODES_OA as a list (diagnostic codes present on arrival) with the Admitting Diagnostic code
df['DIAG_CODES_OA'] = df['ADMITTING_DIAGNOSIS'].str[0:3].apply(lambda x: [x])
# Append to the list values in which the diagnosis is present on arrival "Y"
for i in range(18,68,2):
    mask = df.iloc[:,i+1] != "Y"
    df.loc[mask, df.columns[i]] = ""
    codes = df.iloc[:,i].str[0:3]
    df['DIAG_CODES_OA'] = [
        lst + ([code] if code else [])
        for lst, code in zip(df['DIAG_CODES_OA'], codes)
    ]

In [64]:
# Drop columns for Diagnostic Codes
df = df.drop(columns = df.columns[17:68])

We convert the principal diagnostic codes into one of the 22 chapters as defined in https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2022/icd10cm-tabular-2022-April-1.pdf ,

    
    1. Certain infectious and parasitic diseases (A00-B99)
    2. Neoplasms (C00-D49),
    3. Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism (D50-D89),
    4. Endocrine, nutritional and metabolic diseases (E00-E89),
    5. Mental, Behavioral and Neurodevelopmental disorders (F01-F99),
    6. Diseases of the nervous system (G00-G99),
    7. Diseases of the eye and adnexa (H00-H59),
    8. Diseases of the ear and mastoid process (H60-H95),
    9. Diseases of the circulatory system (I00-I99),
    10. Diseases of the respiratory system (J00-J99),
    11. Diseases of the digestive system (K00-K95),
    12. Diseases of the skin and subcutaneous tissue (L00-L99),
    13. Diseases of the musculoskeletal system and connective tissue (M00-M99),
    14. Diseases of the genitourinary system (N00-N99),
    15. Pregnancy, childbirth and the puerperium (O00-O9A),
    16. Certain conditions originating in the perinatal period (P00-P96),
    17. Congenital malformations, deformations and chromosomal abnormalities (Q00-Q99),
    18. Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified (R00-R99),
    19. Injury, poisoning and certain other consequences of external causes (S00-T88),
    20. External causes of morbidity (V00-Y99),
    21. Factors influencing health status and contact with health services (Z00-Z99),
    22. Codes for special purposes (U00-U85)

In [65]:
# Create a dictionary with Chapter Number as the key, [lower bound of 3 digit codes, upper bound of 3 digit code, description] as the value
code_dict = {1 : ['A00','B99','Certain infectious and parasitic diseases'],
             2 : ['C00','D49','Neoplasms'],
             3 : ['D50','D89','Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism'],
             4 : ['E00','E89','Endocrine, nutritional and metabolic diseases' ],
             5 : ['F01','F99','Mental, Behavioral and Neurodevelopmental disorders' ],
             6 : ['G00','G99', 'Diseases of the nervous system'],
             7 : ['H00','H59','Diseases of the eye and adnexa'],
             8 : ['H60','H95','Diseases of the ear and mastoid process'],
             9 : ['I00','I99','Diseases of the circulatory system'],
             10 : ['J00','J99','Diseases of the respiratory system'],
             11 : ['K00','K95','Diseases of the digestive system'],
             12 : ['L00','L99','Diseases of the skin and subcutaneous tissue'],
             13 : ['M00','M99','Diseases of the musculoskeletal system and connective tissue'],
             14 : ['N00','N99','Diseases of the genitourinary system'],
             15 : ['O00','O9A','Pregnancy, childbirth and the puerperium'],
             16 : ['P00','P96','Certain conditions originating in the perinatal period'],
             17 : ['Q00','Q99','Congenital malformations, deformations and chromosomal abnormalities'],
             18 : ['R00','R99','Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified'],
             19 : ['S00','T88','Injury, poisoning and certain other consequences of external causes'],
             20 : ['V00','Y99','External causes of morbidity'],
             21 : ['Z00','Z99','Factors influencing health status and contact with health services'],
             22 : ['U00','U85','Codes for special purposes']}

# For each chapter, create a new column that runs through the DIAG_CODES_OA list and checks if there is any code between the lower and upper bound lexicographically
for key,value in code_dict.items():
    df['CODE_'+str(key)] = df['DIAG_CODES_OA'].apply(
    lambda codes: int(any(
        isinstance(code, str) and value[0] <= code <= value[1]
        for code in codes
    )))

# Note: Chapter 15 has codes of a different form (e.g. 'O9A'), but this still is captured in the lexicographic ordering

In [66]:
df = df.dropna()

df=df.drop(df[df['PAT_ZIP'] == '`'].index)
df=df.drop(df[df['RACE'] == '`'].index)
df=df.drop(df[df['ETHNICITY'] == '`'].index)
df=df.drop(df[df['PAT_STATUS'] == '`'].index)
df=df.drop(df[df['PAT_AGE'] == '`'].index)

df.PAT_ZIP = df.PAT_ZIP.astype(int)
df.RACE = df.RACE.astype(int)
df.ETHNICITY = df.ETHNICITY.astype(int)
df.PAT_STATUS = df.PAT_STATUS.astype(int)
df.PAT_AGE = df.PAT_AGE.astype(int)


In [67]:
df.info()

<class 'pandas.DataFrame'>
Index: 663667 entries, 0 to 774563
Data columns (total 42 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   RECORD_ID             663667 non-null  int64  
 1   DISCHARGE             663667 non-null  str    
 2   THCIC_ID              663667 non-null  int64  
 3   TYPE_OF_ADMISSION     663667 non-null  float64
 4   SOURCE_OF_ADMISSION   663667 non-null  str    
 5   PAT_STATE             663667 non-null  str    
 6   PAT_ZIP               663667 non-null  int64  
 7   PAT_COUNTRY           663667 non-null  str    
 8   PAT_COUNTY            663667 non-null  float64
 9   PUBLIC_HEALTH_REGION  663667 non-null  float64
 10  PAT_STATUS            663667 non-null  int64  
 11  SEX_CODE              663667 non-null  str    
 12  RACE                  663667 non-null  int64  
 13  ETHNICITY             663667 non-null  int64  
 14  ADMIT_WEEKDAY         663667 non-null  float64
 15  LENGTH_OF_STAY  

In [68]:
df

,RECORD_ID,DISCHARGE,THCIC_ID,TYPE_OF_ADMISSION,SOURCE_OF_ADMISSION,PAT_STATE,PAT_ZIP,PAT_COUNTRY,PAT_COUNTY,PUBLIC_HEALTH_REGION,...,CODE_13,CODE_14,CODE_15,CODE_16,CODE_17,CODE_18,CODE_19,CODE_20,CODE_21,CODE_22
0,120190000375,2019Q1,703003,3.0,2,TX,78418,US,355.0,11.0,...,0,0,0,0,0,0,0,0,0,0
1,120190000376,2019Q1,703003,1.0,1,TX,78335,US,409.0,11.0,...,1,1,0,0,0,1,0,0,0,0
2,120190000377,2019Q1,703003,3.0,2,TX,78336,US,409.0,11.0,...,0,0,0,0,0,0,0,0,0,0
3,120190000378,2019Q1,703003,3.0,2,TX,78411,US,355.0,11.0,...,0,0,0,0,0,0,0,0,0,0
4,120190000379,2019Q1,703003,3.0,2,TX,78414,US,355.0,11.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774554,420186145528,2018Q4,129000,1.0,6,TX,75904,US,5.0,5.0,...,0,1,0,0,0,1,0,0,1,0
774556,420186145532,2018Q4,129000,1.0,1,TX,75964,US,347.0,5.0,...,0,1,0,0,0,1,0,0,0,0
774557,420186145533,2018Q4,129000,1.0,1,TX,75969,US,5.0,5.0,...,0,0,0,0,0,0,1,0,0,0
774562,420186150306,2018Q4,297000,1.0,1,TX,76245,US,181.0,3.0,...,0,0,0,0,0,1,1,0,0,0


In [69]:
df.to_csv('PUDF_base1_1q2019_cleaned.csv', index=False)

Getting rid of the rows with na or ' (invalid) results in around 85% of original data left.